# Nemotron-3-Diarization benchmark (Colab)
**Runtime -> Change runtime type -> T4 GPU**, then run cells top to bottom.

Order matters: do **not** import numpy/torch before the install cell (step 3). The install cell restarts the runtime on purpose; after it restarts, continue from step 4 and **do not re-run steps 2-3**.

## 1. Check the GPU (no Python imports here)

In [ ]:
!nvidia-smi | head -15

## 2. Upload `nemotron_bench.zip`

In [ ]:
from google.colab import files
import zipfile
up = files.upload()   # pick nemotron_bench.zip
zipfile.ZipFile(next(iter(up))).extractall('/content')
!ls -A /content/nemotron_bench

## 3. Install (several minutes), then the runtime restarts by itself
Default is **pyannote only** (fewest conflicts). Set `INSTALL_NEMO = True` only if you also want Nemotron.
Red pip "dependency conflict" lines about `google-adk` / `opentelemetry` can be ignored.

In [ ]:
INSTALL_PYANNOTE = True
INSTALL_NEMO = False   # heavier, more version conflicts; only if you also want Nemotron

import subprocess, os
def sh(c):
    if subprocess.run(c, shell=True).returncode: print('FAILED:', c)
sh('apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1')
sh('pip install -q -U pip setuptools wheel Cython packaging')
sh('pip install -q streamlit pandas pyannote.metrics faster-whisper huggingface_hub')
if INSTALL_PYANNOTE: sh('pip install -q -U pyannote.audio')
if INSTALL_NEMO: sh('pip install -q -U "nemo-toolkit[asr]"')
print('Install finished. Restarting the runtime now (normal). Continue from step 4 afterwards.')
os.kill(os.getpid(), 9)

## 4. After the restart: verify versions (continue here, do not re-run 2-3)

In [ ]:
import os; os.chdir('/content/nemotron_bench')
import numpy, torch
print('numpy', numpy.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Switch the runtime to a T4 GPU'
try:
    import pyannote.audio as pa; print('pyannote.audio', pa.__version__)
except Exception as e: print('pyannote import FAILED:', e)
try:
    import nemo; print('nemo', nemo.__version__)
except Exception: print('NeMo not installed (fine if you only use pyannote)')

## 5. Hugging Face login (token with read access)

In [ ]:
from huggingface_hub import login
login()

## 6. Smoke check: pyannote community-1 loads
**One-time:** open https://huggingface.co/pyannote/speaker-diarization-community-1 while logged in and **accept the conditions**, and make sure the token you used in step 5 has read access. Without that this cell fails with a 401/403 (gated model) error.

In [ ]:
import torch
from pyannote.audio import Pipeline
try:
    pl = Pipeline.from_pretrained('pyannote/speaker-diarization-community-1', token=True)
    pl.to(torch.device('cuda')); print('pyannote pipeline loaded OK'); del pl
except Exception as e:
    print('FAILED:', e, '\nAccept the model conditions on the HF page and re-run step 5 with a valid token.')

### 6b. (Only if you set `INSTALL_NEMO = True`) check Nemotron loads

In [ ]:
from nemo.collections.asr.models import SortformerEncLabelModel
m = SortformerEncLabelModel.from_pretrained('nvidia/Nemotron-3-Diarization').to('cuda').eval()
print('model loaded OK'); del m

## 7. (Optional) Convert ONE big video from Drive
Shared folders must be in **My Drive** first: in Drive, right-click the folder -> Organize -> Add shortcut -> My Drive. Skip this section if your file is already small.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
FOLDER_NAME = 'Psycon_recording'
hits = [p for p in glob.glob('/content/drive/MyDrive/*') if os.path.basename(p).lower() == FOLDER_NAME.lower()]
assert hits, 'Folder not found in My Drive (add a shortcut to it, then re-run).'
SRC = hits[0]
videos = sorted(os.path.join(r, f) for r, _, fs in os.walk(SRC) for f in fs if f.lower().endswith(('.mov', '.mp4')))
for i, v in enumerate(videos):
    print(f'[{i}] {os.path.basename(v)}  {os.path.getsize(v)/1e9:.2f} GB')

In [ ]:
import subprocess

PICK = 3              # <- number from the list above (converts only this one)
MODE = 'audio'        # 'audio' = 16 kHz mono WAV; 'video' = small 480p MP4 (keeps the video player)
CLIP_SECONDS = None   # e.g. 180 for a quick 3-minute test clip

v = videos[PICK]
OUT = '/content/drive/MyDrive/group_discussion_26_converted'
os.makedirs(OUT, exist_ok=True)
stem = os.path.splitext(os.path.basename(v))[0]
clip = ['-t', str(CLIP_SECONDS)] if CLIP_SECONDS else []
if MODE == 'audio':
    out = f'{OUT}/{stem}_16k.wav'
    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-i', v, *clip, '-vn', '-ac', '1', '-ar', '16000', '-c:a', 'pcm_s16le', out]
else:
    out = f'{OUT}/{stem}_small.mp4'
    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-i', v, *clip, '-vf', 'scale=-2:480', '-c:v', 'libx264',
           '-crf', '30', '-preset', 'fast', '-ac', '1', '-ar', '16000', '-c:a', 'aac', out]
print('converting', os.path.basename(v), '...')
subprocess.run(cmd, check=True)
print('->', out, f'{os.path.getsize(out)/1e6:.0f} MB  (download it from Drive, then upload it in the app)')

## 8. Start the app and a public tunnel

In [ ]:
!pip install -q "av<19"

In [ ]:
import subprocess, time, re, os
os.chdir('/content/nemotron_bench')
subprocess.run('pkill -f streamlit; pkill -f cloudflared', shell=True)
subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared', shell=True)
subprocess.Popen('nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > st.log 2>&1 &', shell=True)
time.sleep(8)
subprocess.Popen('nohup ./cloudflared tunnel --url http://localhost:8501 > cf.log 2>&1 &', shell=True)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open('cf.log').read())
    if m: url = m.group(0); break
print('OPEN THIS:', url or 'no URL yet, see cf.log')

In [ ]:
!pip install -q --target /content/cuda12 nvidia-cublas-cu12 nvidia-cudnn-cu12

import subprocess, time, os
os.chdir('/content/nemotron_bench')
subprocess.run('pkill -f streamlit', shell=True)
time.sleep(2)
LD = '/content/cuda12/nvidia/cublas/lib:/content/cuda12/nvidia/cudnn/lib'
subprocess.Popen(f'env LD_LIBRARY_PATH={LD}:$LD_LIBRARY_PATH nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > st.log 2>&1 &', shell=True)
time.sleep(8); print('app restarted')

Open the link. **Anyone with the link can use the app**, so don't share it. If the page errors, check the log:

In [ ]:
!tail -25 /content/nemotron_bench/st.log

## 9. Save results to Drive (Colab wipes files when the session ends). Re-run after each batch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nemotron_runs
!cp -r /content/nemotron_bench/static/runs/. /content/drive/MyDrive/nemotron_runs/
!ls /content/drive/MyDrive/nemotron_runs

## Notes
- In the app, choose **pyannote community-1** and set **exact number of speakers** if you know it (e.g. 8). 0 = automatic.
- Upload the **converted** (small) file in the app, not the original multi-GB video.
- If pyannote fails with 401/403: accept the conditions on its Hugging Face page and re-run step 5 with a token.
- If a NumPy `ufunc ... __module__` error appears, restart the runtime and continue from step 4 (do not re-run the install cell).
- If the GPU runs out of memory, set the ASR model to `medium` in the sidebar.
- The tunnel URL changes each time you re-run step 8.